# 6. Full PID Controller

[Previous lesson](05_derivative_control.ipynb) · [Next lesson](07_controller_tuning.ipynb)

Turn the three control terms into a small reusable object with explicit state, reset behavior, and observable contributions. The goal is reproducibility, not additional control theory.

Work through the explanations, then implement each exercise. Hints and complete reference solutions are collapsed below your code cells. Solutions build in order; Run All does not execute them.

**Prerequisites:** complete the preceding lessons. This notebook starts independently: run the setup cell once before attempting the exercises.

## Previously implemented code

This setup repeats earlier material so you do not need another notebook running. Read it as a recap, then run it. The new work for this lesson remains in the empty cells below.

**Setup inventory:** `motor_dynamics` evaluates the two motor derivatives using SI units and state order `[current, omega]`. Where supplied, `simulate` advances the sampled controller and motor separately, and `plot_run` displays speed, voltage, and term contributions. Controller classes in setup were taught previously; their histories start fresh when you construct an instance. Run setup after restarting the kernel, then execute your exercise cells from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

R, L, K_t, K_e, J, b = 1.0, 0.01, 0.1, 0.1, 0.01, 0.001

def motor_dynamics(t, state, voltage, load_torque):
    current, omega = state
    return [(voltage - R*current - K_e*omega)/L,
            (K_t*current - b*omega - load_torque)/J]
class ThreeTermController:
    def __init__(self, kp, ki, kd):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.integral = 0.0
        self.previous_error = None
    def update(self, setpoint, measurement, dt):
        error = setpoint-measurement
        derivative = 0.0 if self.previous_error is None else (error-self.previous_error)/dt
        self.integral += error*dt
        self.previous_error = error
        self.p_term, self.i_term, self.d_term = self.kp*error, self.ki*self.integral, self.kd*derivative
        return self.p_term+self.i_term+self.d_term
def simulate(controller, target=lambda t: 40.0, load=lambda t: 0.0,
             duration=8.0, dt=0.001, noise=0.0, seed=42):
    # Each row describes the beginning of one control interval.
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("dt must be finite and positive")
    if not np.isfinite(duration) or duration <= 0:
        raise ValueError("duration must be finite and positive")
    if not np.isfinite(noise) or noise < 0:
        raise ValueError("noise must be finite and nonnegative")
    ratio = duration/dt
    if not np.isfinite(ratio):
        raise ValueError("duration/dt must be finite")
    count = round(ratio)
    if count < 1 or not np.isclose(ratio, count, rtol=0, atol=1e-9):
        raise ValueError("duration must be a positive integer multiple of dt")
    rng = np.random.default_rng(seed)
    state = np.array([0.0, 0.0])
    records = []
    for k in range(count):
        t = k*dt
        reference = target(t)
        measured = state[1] + noise*rng.normal()
        voltage = controller.update(reference, measured, dt)
        requested = getattr(controller, "requested", voltage)
        records.append([t, reference, state[0], state[1], measured,
                        requested, voltage, controller.p_term,
                        controller.i_term, controller.d_term])
        result = solve_ivp(motor_dynamics, (t, t+dt), state,
                          args=(voltage, load(t)), rtol=1e-7, atol=1e-9)
        if not result.success:
            raise RuntimeError(result.message)
        state = result.y[:, -1]
    keys = "time target current speed measured requested voltage p i d".split()
    return dict(zip(keys, np.asarray(records).T))

def plot_run(run, title):
    fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 8), layout="constrained")
    axes[0].plot(run['time'], run['target'], '--', label='Target')
    axes[0].plot(run['time'], run['speed'], label='True speed')
    axes[0].set_ylabel('Speed (rad/s)')
    axes[1].plot(run['time'], run['requested'], label='Requested')
    axes[1].plot(run['time'], run['voltage'], '--', label='Applied')
    axes[1].set_ylabel('Voltage (V)')
    for key in ['p', 'i', 'd']:
        axes[2].plot(run['time'], run[key], label=key.upper())
    axes[2].set_ylabel('Contribution (V)')
    axes[2].set_xlabel('Time (s)')
    for ax in axes:
        ax.grid(alpha=.25); ax.legend()
    fig.suptitle(title)
    plt.show()


<img src="assets/lesson-06.svg" alt="Parallel proportional integral and derivative contributions" style="display:block; width:100%; height:auto; margin:0 auto;">

<div class="figure-caption" style="background-color:rgba(128,128,128,0.10);padding:12px 18px;">

**PID structure.** All three paths receive the same error; their voltage contributions are summed. Integral and derivative paths retain history between updates.

</div>

## 6.1 Define the controller contract

A controller should know its gains and history, but not the motor parameters or numerical solver. Its update method receives a target, measurement, and elapsed time and returns one voltage command. Exposed P, I, and D terms let us explain the result.

Keep the unscaled error integral and previous error as state. Reset must clear both plus the displayed terms; otherwise a new experiment inherits an old controller's memory.

### Map physics to an interface

A controller object is useful because some quantities persist between samples. Gains are configuration; the accumulated error and previous sample are history; P/I/D contributions describe the most recent output. Mixing these roles makes reset behavior difficult to reason about.

`update(setpoint, measurement, dt)` returns one voltage. It does not advance the motor or decide when the next sample occurs. The simulation owns time and plant integration. The controller owns only the control law and its history.

`reset()` returns history to a defined startup state while preserving gains. The initial previous error is `None`, not zero: that distinction means “no previous observation.” Reset also clears displayed terms so plots or diagnostic code do not report stale contributions before the next update.

Implement reset first, invoke it from initialization, then implement update in the same order as the earlier arithmetic. This keeps construction and manual reset consistent.

<div class="implementation-task" style="border-left:3px solid #b77979;background-color:rgba(183,121,121,0.08);padding:12px 18px;">

### Your implementation 1 — Initialize and reset history

Create an initial PID class containing only `__init__` and `reset`. Store the three gains, call reset from initialization, and initialize integral, previous_error, and P/I/D terms as described above. Modify the history manually, call reset, and check the gains are preserved while history clears. The next exercise completes this class with update.

</div>

<details>
<summary>Show a hint</summary>

Reset history, not configuration. Call the same reset method during initialization and at experiment boundaries.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
class PID:
    def __init__(self,kp,ki,kd):
        self.kp,self.ki,self.kd=kp,ki,kd
        self.reset()
    def reset(self):
        self.integral=0.0
        self.previous_error=None
        self.p_term=self.i_term=self.d_term=0.0

controller=PID(.4,.5,.01)
controller.integral=8.8
controller.previous_error=3.0
controller.reset()
assert controller.integral==0 and controller.previous_error is None
assert (controller.kp,controller.ki,controller.kd)==(.4,.5,.01)
```

This deliberately incomplete class cannot control the motor yet. The next exercise supplies the full class definition; rerun that definition before constructing controllers for later experiments.

</details>

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 2 — Complete the PID update

Complete the PID class with update(setpoint,measurement,dt), retaining the initialization and reset behavior from the preceding exercise. Rerun the full class definition before creating new instances. Reject nonfinite or nonpositive dt before changing state.

</div>

<details>
<summary>Show a hint</summary>

Use the arithmetic from Lesson 5, moving initialization of history into reset().

</details>

<details>
<summary>Show solution and explanation</summary>

```python
class PID:
    def __init__(self, kp, ki, kd):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.reset()
    def reset(self):
        self.integral = 0.0
        self.previous_error = None
        self.p_term = self.i_term = self.d_term = 0.0
    def update(self, setpoint, measurement, dt):
        if not np.isfinite(dt) or dt <= 0:
            raise ValueError("dt must be finite and positive")
        error = setpoint-measurement
        slope = 0.0 if self.previous_error is None else (error-self.previous_error)/dt
        self.integral += error*dt
        self.previous_error = error
        self.p_term = self.kp*error
        self.i_term = self.ki*self.integral
        self.d_term = self.kd*slope
        return self.p_term+self.i_term+self.d_term
```

This controller can be reused with another plant because it contains no motor-specific variables.

</details>

## 6.2 Check arithmetic and invalid inputs

A small deterministic sequence is better than a plot for catching state errors. At gains $(0.4,0.5,0.01)$, a first error of 10 over 0.1 s gives P=4, I=0.5, D=0. A second error of 8 gives P=3.2, I=0.9, D=−0.2, summing to 3.9 V.

### Why these tests are more informative than a single plot

The first two calls exercise all three terms: the first tests initialization, while the second tests accumulated state and a negative derivative. Check individual contributions as well as the sum. Two wrong terms can accidentally cancel in the total voltage.

A nonpositive time step makes the derivative physically meaningless; zero would also divide by zero. NaN can silently spread through every later result. Reject those values before touching the accumulator or previous error so the next valid call behaves as though the invalid call never happened.

A snapshot of the instance dictionary is sufficient here because the stored values are scalars. This is a deliberately small, inspectable object, not a framework that needs elaborate test infrastructure.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 3 — Test arithmetic and invalid inputs

Check those values, reject dt of 0, −1, and NaN, and verify rejected calls leave the controller state unchanged.

</div>

<details>
<summary>Show a hint</summary>

Snapshot the instance dictionary before invalid calls.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
c=PID(.4,.5,.01)
np.testing.assert_allclose(c.update(10,0,.1),4.5)
np.testing.assert_allclose(c.update(8,0,.1),3.9)
np.testing.assert_allclose([c.p_term,c.i_term,c.d_term],[3.2,.9,-.2])
state=c.__dict__.copy()
for dt in [0,-1,float('nan')]:
    try:c.update(10,0,dt)
    except ValueError:pass
    else:raise AssertionError('Invalid dt accepted')
assert c.__dict__==state
```

Validating before mutation prevents an invalid call from contaminating future outputs.

</details>

## 6.3 Reset between experiments

Reset is an experimental boundary. Two simulations with the same inputs and a reset controller should produce the same outputs. This is different from resetting midway through normal operation, which can abruptly remove the voltage supplied by the integral.

<div class="lesson-callout" style="border-left: 3px solid #b99a62; background-color: rgba(185, 154, 98, 0.08); padding: 12px 18px; margin: 16px 0;">

**Implementation note**

Do not reset at every sample: that would erase the integral and derivative history you are trying to model. Reset once at the beginning of an independent experiment.

</div>

### Reset is not part of every control cycle

Imagine an experiment ending with an integral voltage of 4.4 V. Reusing the controller without reset would start the next run with that stored voltage even if the motor starts from rest. The experiment would no longer match the zero-history assumption.

Conversely, resetting every sample would remove the integral memory and leave no previous error for D. The controller would no longer implement the equations you derived. Call reset at the start of an independent run, not inside the update loop.

A real system may need more sophisticated transfer between operating modes, but that is outside this tutorial. Here the purpose is to make repeated experiments reproducible and to teach exactly which state belongs to the controller.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 4 — Verify repeatable reset

Run one controller twice with reset in between. Check its complete speed and voltage histories match.

</div>

<details>
<summary>Show a hint</summary>

Use the deterministic default measurement with no noise.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
controller=PID(.4,.5,.01)
first=simulate(controller,duration=4)
controller.reset()
assert controller.previous_error is None and controller.integral==0
second=simulate(controller,duration=4)
np.testing.assert_allclose(first['speed'],second['speed'])
np.testing.assert_allclose(first['voltage'],second['voltage'])
plot_run(second,'Reusable PID after reset')
```

Reset makes independent comparisons repeatable while retaining the selected gains.

</details>

## 6.4 Prove the refactor preserves behavior

Refactoring should change organization without changing the control law. Compare the new class with the earlier three-term implementation at identical gains and sample interval. Compare commands as well as speed: the plant can smooth command differences that would otherwise be easy to miss.

### Compare behavior before replacing the reference

Run the old and new controller implementations against identical inputs and compare their full command sequences. Speed alone is insufficient because motor dynamics smooth rapid command differences.

Numerical tolerance accounts for floating-point arithmetic; it should not hide changes in update order or initialization. Both implementations use the same rectangle-rule integral, backward difference, and first-sample convention, so they should agree closely.

After this check, later notebooks can supply the class as previously implemented setup. That makes the code available without making its behavior mysterious: the reader has already derived and tested each term.

<div class="implementation-task" style="border-left: 3px solid #b77979; background-color: rgba(183, 121, 121, 0.08); padding: 12px 18px; margin: 16px 0;">

### Your implementation 5 — Check behavioral equivalence

Run both implementations for four seconds and compare every voltage and speed sample within numerical tolerance.

</div>

<details>
<summary>Show a hint</summary>

Construct both from fresh state; do not reuse the controller from the previous exercise without resetting.

</details>

<details>
<summary>Show solution and explanation</summary>

```python
old=simulate(ThreeTermController(.4,.5,.01),duration=4)
new=simulate(PID(.4,.5,.01),duration=4)
np.testing.assert_allclose(old['voltage'],new['voltage'],rtol=1e-10,atol=1e-10)
np.testing.assert_allclose(old['speed'],new['speed'],rtol=1e-10,atol=1e-10)
print('Refactor preserves the sampled control law.')
```

The class remains ideal: no saturation, anti-windup, or derivative filtering is active yet.

</details>

## What the experiment establishes

The reusable controller reproduces the earlier arithmetic and trajectories. Explicit time steps, reset behavior, and exposed terms make experiments repeatable. The next lesson uses that repeatability to measure performance and compare gains.

## Before moving on

Run your exercises from a fresh kernel in order. Check the stated numerical conditions and explain your plots in your own words: what changed, why it changed, and which model assumptions limit the conclusion. Reference solutions are one implementation, not the only correct approach.

[Previous lesson](05_derivative_control.ipynb) · [Next lesson](07_controller_tuning.ipynb)

## References and further reading

[Åström and Murray, Feedback Systems, Chapter 11](https://www.cds.caltech.edu/~murray/books/AM08/pdf/fbs-pid_24Jul2020.pdf) — additional background on PID structure and implementation. Our examples use independent parallel gains and the conventions stated in this lesson.